<a href="https://colab.research.google.com/github/AizenSosuke2025/RecommendationSystem/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import gc

file_list = [
    'NetflixPrizeDataset/combined_data_1.txt',
    'NetflixPrizeDataset/combined_data_2.txt',
    'NetflixPrizeDataset/combined_data_3.txt',
    'NetflixPrizeDataset/combined_data_4.txt'
]
global_rating_distribution = pd.Series(dtype='int64')
movie_popularity = {}
user_activity = {}
total_ratings = 0
unique_users = set()
unique_movies = set()

for file in file_list:
    print(f"Loading {file}...")

    # The 'Date' column is dropped immediately since we don't need it for this specific EDA.
    df = pd.read_csv(file, header=None, names=['Cust_Id', 'Rating'], usecols=[0, 1])

    # Rows with Movie IDs have a NaN rating and a colon in the Cust_Id (e.g., "1:")
    movie_mask = df['Rating'].isna()

    # Extract the movie ID, remove the colon, and forward-fill it down the rows
    df.loc[movie_mask, 'Movie_Id'] = df.loc[movie_mask, 'Cust_Id'].str.replace(':', '')
    df['Movie_Id'] = df['Movie_Id'].ffill().astype('int16') # int16 covers up to 32k movies

    # Drop the original Movie ID rows, leaving only actual user ratings
    df = df.dropna()

    # Downcast the remaining data
    df['Cust_Id'] = df['Cust_Id'].astype('int32')
    df['Rating'] = df['Rating'].astype('int8')
    print(f"Aggregating data for {file}...")

    # Aggregate Rating Distribution
    global_rating_distribution = global_rating_distribution.add(df['Rating'].value_counts(), fill_value=0)

    # Aggregate Movie Popularity
    local_movie_counts = df['Movie_Id'].value_counts().to_dict()
    for movie, count in local_movie_counts.items():
        movie_popularity[movie] = movie_popularity.get(movie, 0) + count

    # Aggregate User Activity
    local_user_counts = df['Cust_Id'].value_counts().to_dict()
    for user, count in local_user_counts.items():
        user_activity[user] = user_activity.get(user, 0) + count

    # Track totals for Sparsity Calculation
    total_ratings += len(df)
    unique_users.update(df['Cust_Id'].unique())
    unique_movies.update(df['Movie_Id'].unique())

    del df, local_movie_counts, local_user_counts
    gc.collect()
    print(f"Finished {file}. RAM cleared.\n")

print("Data processing complete!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

movie_popularity_series = pd.Series(movie_popularity).sort_values(ascending=False)
user_activity_series = pd.Series(user_activity).sort_values(ascending=False)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Rating Distribution
x_vals = global_rating_distribution.index.astype(int)
y_vals = global_rating_distribution.values

sns.barplot(x=x_vals, y=y_vals, hue=x_vals, palette='viridis', legend=True, ax=axes[0])
axes[0].set_title('Distribution of Ratings')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Total Count')
axes[0].legend(title="Star Rating", loc='upper left')

# Content Popularity Trends
axes[1].plot(movie_popularity_series.values, color='purple')
axes[1].fill_between(range(len(movie_popularity_series)), movie_popularity_series.values, alpha=0.3, color='purple')
axes[1].set_title('Content Popularity (Ratings per Movie)')
axes[1].set_xlabel('Movie ID (Ranked by Popularity)')
axes[1].set_ylabel('Number of Ratings')

# User Activity Patterns
axes[2].plot(user_activity_series.values, color='teal')
axes[2].fill_between(range(len(user_activity_series)), user_activity_series.values, alpha=0.3, color='teal')
axes[2].set_title('User Activity (Ratings per User)')
axes[2].set_xlabel('User ID (Ranked by Activity)')
axes[2].set_ylabel('Number of Ratings')

plt.tight_layout()
plt.show()

# Data Sparsity & Final Insights
num_users = len(unique_users)
num_movies = len(unique_movies)
possible_ratings = num_users * num_movies
sparsity = (1.0 - (total_ratings / possible_ratings)) * 100

print("\n" + "="*40)
print(" EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*40)
print(f"Total Valid Ratings: {total_ratings:,}")
print(f"Total Unique Users:  {num_users:,}")
print(f"Total Unique Movies: {num_movies:,}")
print(f"Data Matrix Sparsity: {sparsity:.4f}%")
print("-" * 40)
print(f"1. Most rated movie has {movie_popularity_series.max():,} ratings.")
print(f"2. Least rated movie has {movie_popularity_series.min():,} ratings.")
print(f"3. The most active user gave {user_activity_series.max():,} ratings.")
print(f"4. The median user gave {user_activity_series.median():.0f} ratings.")

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'

def parse_probe_file(file_path):
    print("Parsing probe.txt into memory...")
    probe_dict = {}
    current_movie = None

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: # Skip empty lines
                continue
            if line.endswith(':'):
                current_movie = int(line[:-1])
                probe_dict[current_movie] = set()
            else:
                probe_dict[current_movie].add(int(line))

    print(f"Successfully parsed probe pairs for {len(probe_dict)} movies.\n")
    return probe_dict
probe_lookup = parse_probe_file(base_path + 'probe.txt')

file_list = [
    base_path + 'combined_data_1.txt',
    base_path + 'combined_data_2.txt',
    base_path + 'combined_data_3.txt',
    base_path + 'combined_data_4.txt'
]

print("Creating CSV files")
train_file = open(base_path + 'train_ratings.csv', 'w')
probe_file = open(base_path + 'probe_ratings.csv', 'w')

train_file.write("Movie_Id,Cust_Id,Rating\n")
probe_file.write("Movie_Id,Cust_Id,Rating\n")

for file in file_list:
    if not os.path.exists(file):
        print(f"WARNING: {file} not found. Check your file name/path. Skipping.")
        continue

    print(f"Splitting {file.split('/')[-1]} using official probe guidelines...")
    current_movie_id = None

    with open(file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(':'):
                current_movie_id = int(line[:-1])
            else:
                # Parse Customer ID and Rating
                parts = line.split(',')
                cust_id = int(parts[0])
                rating = int(parts[1])

                # Check if this pair belongs to the official test set (probe)
                if current_movie_id in probe_lookup and cust_id in probe_lookup[current_movie_id]:
                    probe_file.write(f"{current_movie_id},{cust_id},{rating}\n")
                else:
                    train_file.write(f"{current_movie_id},{cust_id},{rating}\n")

train_file.close()
probe_file.close()

## Content-Based Filtering

In [ ]:
import pandas as pd
import numpy as np
import math
import os
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'

K = 10
RATING_THRESHOLD = 4
NUM_USERS_TO_SAMPLE = 5000

print("1. Loading movie titles and building TF-IDF matrix...")
movies = pd.read_csv(base_path + 'movie_titles.csv', encoding='ISO-8859-1', header=None, names=['Movie_Id', 'Year', 'Name'], on_bad_lines='skip')
movies['Name'] = movies['Name'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['Name'])
total_catalog_size = tfidf_matrix.shape[0]
movie_id_to_idx = pd.Series(movies.index, index=movies['Movie_Id']).to_dict()

print("2. Loading probe evaluation dataset and sampling users...")
test_df = pd.read_csv(base_path + 'probe_ratings.csv')
test_df = test_df[test_df['Movie_Id'].isin(movie_id_to_idx.keys())].copy()
test_df['Matrix_Idx'] = test_df['Movie_Id'].map(movie_id_to_idx)

# Find users who have enough test items to calculate ranking metrics
probe_user_counts = test_df['Cust_Id'].value_counts()
eligible_users = probe_user_counts[probe_user_counts >= 5].index.tolist()

# Randomly pick a small group of users from the eligible pool
random.seed(42) # Keeps the random selection the same every time you run it
sampled_users = set(random.sample(eligible_users, min(NUM_USERS_TO_SAMPLE, len(eligible_users))))

test_df = test_df[test_df['Cust_Id'].isin(sampled_users)]
print(f"Targeting evaluation for a SUBSET of {len(sampled_users)} users (out of {len(eligible_users)} total eligible).")

print("3. Streaming and extracting training records for sampled users...")
train_chunks = []

# Using tqdm to show progress
for chunk in tqdm(pd.read_csv(base_path + 'train_ratings.csv', chunksize=10_000_000), total=10, desc="Reading Chunks"):
    filtered_chunk = chunk[chunk['Cust_Id'].isin(sampled_users)]
    if not filtered_chunk.empty:
        train_chunks.append(filtered_chunk)

print("\nConcatenating training data...")
train_df = pd.concat(train_chunks, ignore_index=True)
train_df = train_df[train_df['Movie_Id'].isin(movie_id_to_idx.keys())].copy()
train_df['Matrix_Idx'] = train_df['Movie_Id'].map(movie_id_to_idx)

print("4. Grouping data by user for O(1) lookups...")
train_groups = dict(list(train_df.groupby('Cust_Id')))
test_groups = dict(list(test_df.groupby('Cust_Id')))

import gc
del train_df, test_df, train_chunks
gc.collect()

print("5. Starting subset evaluation loop...")

# METRIC FUNCTIONS
def calculate_ndcg(actual_relevant, predicted_top_k, k):
    top_k = predicted_top_k[:k]
    dcg = sum([1.0 / math.log2(i + 2) for i, item in enumerate(top_k) if item in actual_relevant])
    idcg = sum([1.0 / math.log2(i + 2) for i in range(min(len(actual_relevant), k))])
    return dcg / idcg if idcg > 0 else 0.0

def calculate_ild(top_k_indices, tfidf_mat):
    if len(top_k_indices) <= 1: return 1.0
    sim_matrix = linear_kernel(tfidf_mat[top_k_indices], tfidf_mat[top_k_indices])
    distances = [1.0 - sim_matrix[i, j] for i in range(len(top_k_indices)) for j in range(i + 2, len(top_k_indices))]
    return np.mean(distances) if distances else 1.0

all_precisions, all_recalls, all_hit_rates, all_ndcgs, all_diversities = [], [], [], [], []
all_actual_ratings, all_predicted_ratings = [], []
system_wide_recommendations = set()

# Process users
user_count = 0
for user in sampled_users:
    user_count += 1
    if user_count % 100 == 0:
        print(f"Progress: Evaluated {user_count}/{len(sampled_users)} users...")

    u_train = train_groups.get(user)
    u_test = test_groups.get(user)
    if u_train is None or u_test is None: continue

    liked_train = u_train[u_train['Rating'] >= RATING_THRESHOLD]
    if len(liked_train) == 0: continue

    # Generate profile
    user_profile = np.zeros(total_catalog_size)
    for idx in liked_train['Matrix_Idx']:
        user_profile += linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    user_profile[u_train['Matrix_Idx'].tolist()] = -1

    # Top-K Recommendations
    predicted_top_k = user_profile.argsort()[-K:][::-1].tolist()
    system_wide_recommendations.update(predicted_top_k)

    # MAE & RMSE Predictions
    for _, row in u_test.iterrows():
        test_movie_idx = row['Matrix_Idx']
        sims = linear_kernel(tfidf_matrix[test_movie_idx], tfidf_matrix[liked_train['Matrix_Idx'].tolist()]).flatten()
        if sims.sum() > 0:
            pred_rating = np.dot(sims, liked_train['Rating'].values) / sims.sum()
            all_actual_ratings.append(row['Rating'])
            all_predicted_ratings.append(pred_rating)

    # Ranking metrics
    actual_relevant_items = set(u_test[u_test['Rating'] >= RATING_THRESHOLD]['Matrix_Idx'].tolist())
    if len(actual_relevant_items) > 0:
        hits = set(predicted_top_k).intersection(actual_relevant_items)
        all_precisions.append(len(hits) / K)
        all_recalls.append(len(hits) / len(actual_relevant_items))
        all_hit_rates.append(1 if len(hits) > 0 else 0)
        all_ndcgs.append(calculate_ndcg(actual_relevant_items, predicted_top_k, K))

    all_diversities.append(calculate_ild(predicted_top_k, tfidf_matrix))

coverage = len(system_wide_recommendations) / total_catalog_size
print("\n" + "="*50)
print(f" COMPLETED SUBSET EVALUATION ({len(sampled_users)} Users, K={K})")
print("="*50)
if all_predicted_ratings:
    print(f"Mean Absolute Error (MAE):     {mean_absolute_error(all_actual_ratings, all_predicted_ratings):.4f}")
    rmse = np.sqrt(mean_squared_error(all_actual_ratings, all_predicted_ratings))
    print(f"Root Mean Squared Error (RMSE):{rmse:.4f}")

print(f"Precision@{K}:                  {np.mean(all_precisions)*100:.2f}%")
print(f"Recall@{K}:                     {np.mean(all_recalls)*100:.2f}%")
print(f"Hit Rate:                      {np.mean(all_hit_rates)*100:.2f}%")
print(f"NDCG@{K}:                       {np.mean(all_ndcgs):.4f}")
print(f"Catalog Coverage:              {coverage*100:.2f}%")
print(f"Intra-List Diversity (ILD):    {np.mean(all_diversities):.4f}")
print("="*50)